# Докрукс в Google Colab

Ноутбук запускает сервис на бесплатной машине Colab без видеокарты:

1. ставит Tesseract и сам сервис;
2. распознаёт открытый набор и сверяет результат со скриптом организаторов;
3. собирает `results.json` для любого другого набора в том же формате;
4. показывает разбор одного документа с подсветкой полей.

Выполняйте ячейки по порядку. Установка занимает около двух минут.

## 1. Установка

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y tesseract-ocr tesseract-ocr-rus > /dev/null
mkdir -p /opt/tessdata_best
wget -q -O /opt/tessdata_best/rus.traineddata   https://github.com/tesseract-ocr/tessdata_best/raw/e12c65a915945e4c28e237a9b52bc4a8f39a0cec/rus.traineddata
echo "b617eb6830ffabaaa795dd87ea7fd251adfe9cf0efe05eb9a2e8128b7728d6b6  /opt/tessdata_best/rus.traineddata" | sha256sum -c -
rm -rf /content/docrux
git clone -q --depth 1 https://github.com/dimon0804/Docrux.git /content/docrux
pip install -q /content/docrux/backend
tesseract --version | head -1

In [ ]:
import os

os.environ["DOCRUX_TESSDATA"] = "/opt/tessdata_best"
os.environ["DOCRUX_CONFIG_DIR"] = "/content/docrux/config"
ROOT = "/content/docrux"

## 2. Открытый набор

Сервис читает 30 изображений, отвечает на 120 вопросов и сохраняет `results.json`. Затем скрипт организаторов сравнивает его с эталоном.

In [ ]:
!docrux batch {ROOT}/data/open/documents.json --out /content/results_open.json 2>/dev/null
!python {ROOT}/data/open/evaluate.py --reference {ROOT}/data/open/reference_results.json --predictions /content/results_open.json | head -9

## 3. Свой набор

Загрузите архив с набором в том же формате: `documents.json`, `questions.json` и папка `documents`. Ячейка распакует его и соберёт `results.json`. Если в архиве есть `reference_results.json`, результат будет сразу проверен.

In [ ]:
import glob
import zipfile

from google.colab import files

uploaded = files.upload()
archive = next(iter(uploaded))
target = "/content/dataset"
!rm -rf {target}
with zipfile.ZipFile(archive) as zf:
    zf.extractall(target)

manifest = sorted(glob.glob(f"{target}/**/documents.json", recursive=True))[0]
folder = os.path.dirname(manifest)
print("Набор:", folder)

!docrux batch {manifest} --out /content/results.json 2>/dev/null
if os.path.exists(f"{folder}/reference_results.json"):
    !python {ROOT}/data/open/evaluate.py --reference {folder}/reference_results.json --predictions /content/results.json | head -9
files.download("/content/results.json")

## 4. Один документ

Можно взять пример из открытого набора или загрузить свой файл. Зелёные рамки показывают, где на снимке найдено каждое поле.

In [ ]:
import cv2
from IPython.display import display
from PIL import Image

from docrux.pipeline import Pipeline

pipeline = Pipeline.default()


def show(path, question=None, doc_type=None):
    processed = pipeline.analyze(open(path, "rb").read(), doc_type)
    analysis = processed.analysis
    print("Тип:", analysis.document_title, "(определён автоматически)" if doc_type is None else "")
    for field in analysis.fields:
        print(f"  {field.label}: {field.value}")
    print()
    print(pipeline.check(analysis).answer)
    if question:
        answer = pipeline.ask(analysis, question)
        print()
        print("Вопрос:", question)
        print("Ответ:", answer.answer)
        print("Источники:", ", ".join(answer.sources) or "нет")

    image = processed.image.copy()
    height, width = image.shape[:2]
    for field in analysis.fields:
        if field.box is None:
            continue
        x0, y0 = int(field.box.x * width), int(field.box.y * height)
        x1, y1 = int((field.box.x + field.box.w) * width), int((field.box.y + field.box.h) * height)
        cv2.rectangle(image, (x0 - 4, y0 - 4), (x1 + 4, y1 + 4), (72, 184, 80), 3)
    preview = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    preview.thumbnail((900, 1200))
    display(preview)


show(f"{ROOT}/data/open/documents/open-002.png", "Какова дата выдачи?")

In [ ]:
show(f"{ROOT}/data/open/documents/open-030.png", "Какой размер стипендии положен студенту?")

Свой файл: JPG, PNG или PDF.

In [ ]:
uploaded = files.upload()
show(next(iter(uploaded)), "Содержит ли документ все обязательные сведения?")

## 5. Плохой снимок

Тот же документ после наклона, тени и сжатия. Сервис сначала выравнивает снимок, потом читает.

In [ ]:
import random
import sys

sys.path.insert(0, f"{ROOT}/tools")
import numpy as np
from degrade import photo

source = cv2.imread(f"{ROOT}/data/open/documents/open-021.png")
damaged = photo(source, random.Random(3))
cv2.imwrite("/content/photo.jpg", damaged)
before = Image.fromarray(cv2.cvtColor(damaged, cv2.COLOR_BGR2RGB))
before.thumbnail((500, 700))
display(before)
show("/content/photo.jpg")